In [2]:
import sys
import os

# Define the project root directory.
# Based on the file list, '/content/drive/MyDrive/capstone project/'
# seems to be the most likely candidate for your project root.
project_root = '/content/drive/MyDrive/capstone project/'

# Add the project root to sys.path so Python can find your 'app' module
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Added '{project_root}' to sys.path.")
print(f"Current sys.path: {sys.path}")

# Optional: Verify if 'app/main.py' or 'main.py' exists within the project_root
# This is a check to ensure the file structure matches the import statement
app_main_path = os.path.join(project_root, 'app', 'main.py')
root_main_path = os.path.join(project_root, 'main.py')

if not os.path.exists(app_main_path) and not os.path.exists(root_main_path):
    print("Warning: Neither 'app/main.py' nor 'main.py' found in the specified project root. "
          "You might need to adjust the `project_root` variable or your file structure.")
else:
    print("Found application main file(s).")


Added '/content/drive/MyDrive/capstone project/' to sys.path.
Current sys.path: ['/content/drive/MyDrive/capstone project/', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
Found application main file(s).


In [5]:
import pytest
from fastapi.testclient import TestClient
from app.main import app

# Initialize the test client
client = TestClient(app)

### Test Case 1: Health Check Endpoint
def test_health_check():
    """
    GIVEN the FastAPI application is running
    WHEN the GET /health endpoint is called
    THEN it should return a 200 OK status and {"status": "ok"}
    """
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json() == {"status": "ok"}


### Test Case 2: Single Prediction Success & Payload Verification
def test_predict_single_success():
    """
    GIVEN a valid customer feature payload
    WHEN the POST /predict endpoint is called
    THEN it should return a 200 OK status and a structured prediction response
    """
    payload = {
        "tenure_months": 12.5,
        "support_tickets": 5,
        "monthly_charges": 79.99,
        "monthly_active_days": 3
    }
    response = client.post("/predict", json=payload)

    assert response.status_code == 200

    json_data = response.json()
    # Verify all required response keys exist
    assert "churn_probability" in json_data
    assert "predicted_class" in json_data
    assert "risk_level" in json_data
    assert "risk_explanation" in json_data

    # Verify data types and constraints
    assert isinstance(json_data["churn_probability"], float)
    assert json_data["predicted_class"] in [0, 1]
    assert json_data["risk_level"] in ["low", "medium", "high"]
    assert isinstance(json_data["risk_explanation"], str)


### Test Case 3: Input Validation Constraints (Pydantic Enforcement)
def test_predict_validation_error():
    """
    GIVEN an invalid customer feature payload (out-of-bounds metrics)
    WHEN the POST /predict endpoint is called
    THEN it should trigger Pydantic validation and return a 422 Unprocessable Entity
    """
    # 'support_tickets' cannot be negative, 'monthly_active_days' cannot exceed 31
    invalid_payload = {
        "tenure_months": 8.0,
        "support_tickets": -3,
        "monthly_charges": 45.00,
        "monthly_active_days": 42
    }
    response = client.post("/predict", json=invalid_payload)

    assert response.status_code == 422
    assert "detail" in response.json()


### Test Case 4 (Bonus): Batch Prediction Verification
def test_predict_batch_success():
    """
    GIVEN a list of multiple valid customer feature payloads
    WHEN the POST /batch_predict endpoint is called
    THEN it should return a 200 OK status and predictions matching the input length
    """
    batch_payload = [
        {
            "tenure_months": 24.0,
            "support_tickets": 1,
            "monthly_charges": 50.00,
            "monthly_active_days": 28
        },
        {
            "tenure_months": 2.0,
            "support_tickets": 8,
            "monthly_charges": 120.00,
            "monthly_active_days": 1
        }
    ]
    response = client.post("/batch_predict", json=batch_payload)

    assert response.status_code == 200

    json_data = response.json()
    assert "predictions" in json_data
    assert len(json_data["predictions"]) == 2

In [3]:
import os
import shutil

project_root = '/content/drive/MyDrive/capstone project/'
app_dir = os.path.join(project_root, 'app')
source_main_path = os.path.join(project_root, 'main.py')
dest_main_path = os.path.join(app_dir, 'main.py')

# Create the 'app' directory if it doesn't exist
if not os.path.exists(app_dir):
    os.makedirs(app_dir)
    print(f"Created directory: {app_dir}")
else:
    print(f"Directory already exists: {app_dir}")

# Move main.py into the 'app' directory
if os.path.exists(source_main_path) and not os.path.exists(dest_main_path):
    shutil.move(source_main_path, dest_main_path)
    print(f"Moved '{source_main_path}' to '{dest_main_path}'")
elif os.path.exists(dest_main_path):
    print(f"'{dest_main_path}' already exists. No file moved.")
else:
    print(f"Warning: '{source_main_path}' not found. Please ensure your main FastAPI application file is named 'main.py' and located in the project root before moving.")

# Also create an __init__.py file to make 'app' a package
init_file_path = os.path.join(app_dir, '__init__.py')
if not os.path.exists(init_file_path):
    with open(init_file_path, 'w') as f:
        pass # Create an empty __init__.py
    print(f"Created empty __init__.py at: {init_file_path}")
else:
    print(f"'{init_file_path}' already exists.")

print("File structure adjusted to resolve 'ModuleNotFoundError'.")

Created directory: /content/drive/MyDrive/capstone project/app
Moved '/content/drive/MyDrive/capstone project/main.py' to '/content/drive/MyDrive/capstone project/app/main.py'
Created empty __init__.py at: /content/drive/MyDrive/capstone project/app/__init__.py
File structure adjusted to resolve 'ModuleNotFoundError'.
